In [ ]:
import json
import heapq
import numpy as np
import networkx as nx
from scipy.spatial import KDTree


def generate_points(N, seed, min_dist, min_dist_decay):
    np.random.seed(seed)
    points = []

    while len(points) < N:
        candidate = np.random.rand(2)
        if all(np.linalg.norm(candidate - point) >= min_dist for point in points):
            points.append(candidate)
        min_dist *= min_dist_decay

    #  return np.array(points)
    return np.array([[x[0]*500, x[1]*500] for x in points])


def create_hierarchical_knn_graph(
    n, k, hubs_percentage, hub_connection_bidirectional_percentage, seed, min_dist = 0.2, min_dist_decay = 0.995
):
    points = generate_points(n, seed, min_dist, min_dist_decay)
    G = nx.DiGraph()
    tree = KDTree(points)
    np.random.seed(
        seed
    )  # Ensure reproducibility for hub selection and bidirectional decision
    num_hubs = max(1, int(len(points) * hubs_percentage))
    hubs = np.random.choice(range(len(points)), size=num_hubs, replace=False)

    # Adding bidirectional edges for k-nearest neighbors
    for i, point in enumerate(points):
        G.add_node(f"node{i}", pos=(point[0], point[1]))
        distances, indices = tree.query(
            point, k + 1
        )  # k+1 because the point itself is included
        for j in range(
            1, len(indices)
        ):  # skip the first index because it is the point itself
            if not G.has_edge(f"node{i}", f"node{indices[j]}"):
                G.add_edge(f"node{i}", f"node{indices[j]}", weight=distances[j])
            if not G.has_edge(f"node{indices[j]}", f"node{i}"):
                G.add_edge(f"node{indices[j]}", f"node{i}", weight=distances[j])

    # Adding hub connections with a chance of being bidirectional
    for hub in hubs:
        distances, indices = tree.query(
            points[hub], k + num_hubs
        )  # More connections for hubs
        for j in range(
            1, len(indices)
        ):  # skip the first index because it is the point itself
            if np.random.rand() < hub_connection_bidirectional_percentage:
                if not G.has_edge(f"node{hub}", f"node{indices[j]}"):
                    G.add_edge(f"node{hub}", f"node{indices[j]}", weight=distances[j])
                if not G.has_edge(f"node{indices[j]}", f"node{hub}"):
                    G.add_edge(f"node{indices[j]}", f"node{hub}", weight=distances[j])
            else:
                if np.random.rand() < 0.5:
                    if not G.has_edge(f"node{hub}", f"node{indices[j]}"):
                        G.add_edge(
                            f"node{hub}", f"node{indices[j]}", weight=distances[j]
                        )
                else:
                    if not G.has_edge(f"node{indices[j]}", f"node{hub}"):
                        G.add_edge(
                            f"node{indices[j]}", f"node{hub}", weight=distances[j]
                        )

    return G


# Parameters for graph generation
nodes = 20
k = 3
hubs_percentage = 0.2
hub_connection_bidirectional_percentage = 0.1
seed = 0

G = create_hierarchical_knn_graph(
    nodes, k, hubs_percentage, hub_connection_bidirectional_percentage, seed
)

def shortest_paths_with_forbidden(G, F, F_4):
    forbidden_sets = [set(path) for path in F] + [set(path) for path in F_4]
    
    def is_path_allowed(path):
        for forbidden_set in forbidden_sets:
            if forbidden_set.issubset(set(path)):
                return False
        return True

    def dijkstra_with_forbidden(G, source):
        # Initialization
        dist = {node: float('inf') for node in G.nodes()}
        dist[source] = 0
        paths = {node: [] for node in G.nodes()}
        paths[source] = [source]
        queue = [(0, source)]
        visited = set()
        
        while queue:
            current_dist, current_node = heapq.heappop(queue)
            
            if current_node in visited:
                continue
            
            visited.add(current_node)
            
            for neighbor in G.neighbors(current_node):
                weight = G[current_node][neighbor].get('weight', 1)
                distance = current_dist + weight
                
                if distance < dist[neighbor]:
                    # Check if the path is allowed
                    new_path = paths[current_node] + [neighbor]
                    if is_path_allowed(new_path):
                        dist[neighbor] = distance
                        paths[neighbor] = new_path
                        heapq.heappush(queue, (distance, neighbor))
        
        del dist[source]
        del paths[source]
        return dist, paths

    all_pairs_shortest_path_length = {}
    all_pairs_shortest_paths = {}
    
    for source in G.nodes():
        dists, paths = dijkstra_with_forbidden(G, source)
        all_pairs_shortest_path_length[source] = dists
        all_pairs_shortest_paths[source] = paths
    
    return all_pairs_shortest_path_length, all_pairs_shortest_paths

def visualize_graph_with_edge_colors(G):
    import matplotlib.pyplot as plt

    pos = nx.get_node_attributes(G, "pos")
    edge_colors = []

    for u, v in G.edges():
        if G.has_edge(v, u):
            edge_colors.append("red")  # Bidirectional edge
        else:
            edge_colors.append("green")  # Unidirectional edge

    plt.figure(figsize=(10, 10))
    nx.draw(
        G,
        pos,
        with_labels=True,
        node_size=300,
        node_color="skyblue",
        edge_color=edge_colors,
        width=2,
    )
    plt.show()


visualize_graph_with_edge_colors(G)

In [ ]:
np.random.seed(seed)

E_P = [] # two-way block sides
A_P = [] # one-way block sides
for u, v in G.edges():
    if G.has_edge(v, u):
        E_P.append((u, v))
    else:
        A_P.append((u, v))


def find_all_paths_of_length_k(G, k):
    def dfs(current_node, path, k):
        if len(path) == k:
            if len(set(path)) == k:
                all_paths.append(path)
            return
        for neighbor in G.neighbors(current_node):
            if neighbor not in path:  # Avoid cycles
                dfs(neighbor, path + [neighbor], k)
    all_paths = []
    for node in G.nodes():
        dfs(node, [node], k)
    return all_paths

In [ ]:
NUM_FORBIDDEN_NODES = nodes // 5
NUM_FORBIDDEN_FOUR_NODES = nodes // 6

np.random.seed(seed)
print("Looking for F and F_4 and computing all distances",end="", flush=True)
all_paths_of_length_3 = find_all_paths_of_length_k(G, 3)
all_paths_of_length_4 = find_all_paths_of_length_k(G, 4)
while True:
    F = [all_paths_of_length_3[i] for i in np.random.choice(len(all_paths_of_length_3), NUM_FORBIDDEN_NODES, replace=False)]
    F_4 = [all_paths_of_length_4[i] for i in np.random.choice(len(all_paths_of_length_4), NUM_FORBIDDEN_FOUR_NODES, replace=False)]
    gamma, gamma_paths = shortest_paths_with_forbidden(G, F, F_4)
    if np.max([np.max(list(v.values())) for v in gamma.values()]) != np.inf:
        print()
        break
    print(end=".", flush=True)

s, t = E_P[np.random.randint(0, len(E_P))]
s, t

L = [e for e in G.edges if G[e[0]][e[1]]["weight"] > 130]

L_e = {e: [] for e in L}

for e in L:
    u, v = e
    for start_node, paths in gamma_paths.items():
        for end_node, path in paths.items():
            if any((path[i], path[i + 1]) == e for i in range(len(path) - 1)):
                L_e[e].append((start_node, end_node))

In [ ]:
V_ij = {(i,j): [] for i in G.nodes for j in G.nodes if i != j}

for i in G.nodes:
    for j in G.nodes:
        if i == j:
            continue
        u = gamma_paths[i][j][-2]
        for k in G.nodes:
            if j == k:
                continue
            v = gamma_paths[j][k][1]
            if (u,j,v) in F:
                print("OMG")
                V_ij[(i,j)].append(k)
        #  break
# TODO: V_ij is currently dictionary of empty lists

In [ ]:
B_delta = {tuple(delta): [] for delta in F_4}

for u_1, u_2, u_3, u_4 in F_4:
    for v in G.nodes:
        if v == u_2:
            continue
        if gamma_paths[v][u_2][-2:] == [u_1, u_2]:
            B_delta[(u_1, u_2, u_3, u_4)].append(v)

B_delta

In [ ]:
B_delta_hat = {tuple(delta): [] for delta in F_4}

for u_1, u_2, u_3, u_4 in F_4:
    for v in G.nodes:
        if v == u_3:
            continue
        if gamma_paths[v][u_3][-3:] == [u_1, u_2, u_3]:
            B_delta_hat[(u_1, u_2, u_3, u_4)].append(v)

B_delta_hat

In [ ]:
E_delta = {tuple(delta): [] for delta in F_4}

for u_1, u_2, u_3, u_4 in F_4:
    for v in G.nodes:
        if v == u_3:
            continue
        if gamma_paths[u_3][v][:2] == [u_3, u_4]:
            E_delta[(u_1, u_2, u_3, u_4)].append(v)

E_delta

In [ ]:
E_delta_hat = {tuple(delta): [] for delta in F_4}

for u_1, u_2, u_3, u_4 in F_4:
    for v in G.nodes:
        if v == u_2:
            continue
        if gamma_paths[u_2][v][:3] == [u_2, u_3, u_4]:
            E_delta_hat[(u_1, u_2, u_3, u_4)].append(v)

E_delta_hat

In [ ]:
ell_nice = {}
for u in G.nodes:
    ell_nice[u] = {}
    for v in G.neighbors(u):
        ell_nice[u][v] = G[u][v]["weight"]

In [ ]:
data = {
    "A_P": A_P,
    "E_P": E_P,
    #  "ell": ell_nice,
    "ell": gamma,
    "F": F,
    "gamma": gamma,
    "s": s,
    "t": t,
    "V_T": list(G.nodes),
    "L": L,
    "L_e": {str(k): v for k, v in L_e.items()},
    "V_ij": {str(k): v for k, v in V_ij.items()},
    "F_4": F_4,
    "B_delta": {str(k): v for k, v in B_delta.items()},
    "hat_B_delta": {str(k): v for k, v in B_delta_hat.items()},
    "E_delta": {str(k): v for k, v in E_delta.items()},
    "hat_E_delta": {str(k): v for k, v in E_delta_hat.items()}
}

# Save data to JSON
with open("data.json", "w") as f:
    json.dump(data, f, indent=4)

json.dumps(data)